## Enriching stock market data using Open AI API 

<p align="center">
    <img src="images/nasdaq100.png" width="450">
</p>

The Nasdaq-100 is a stock market index made up of 101 equity securities issued by 100 of the largest non-financial companies listed on the Nasdaq stock exchange. It helps investors compare stock prices with previous prices to determine market performance.

In this project you are provided with two CSV files containing Nasdaq-100 stock information:
- _**nasdaq100_CA.csv**_: contains information about companies in the index such as symbol, name, etc. For this analysis, only companies headquartered in California have been selected.
- _**nasdaq100_price_change.csv**_: contains price changes per stock across periods including (but not limited to) one day, five days, one month, six months, one year, etc.

As an AI developer, you will leverage the OpenAI API to classify companies into sectors and produce a summary of sector and company performance for this year, for the companies in the index that are headquartered in California.

# CSV with Nasdaq-100 stock data

In this project, you have available two CSV files `nasdaq100_CA.csv` and `nasdaq100_price_change.csv`.

## nasdaq100_CA.csv

```py
symbol,name,headQuarter,dateFirstAdded,cik,founded
AAPL,Apple Inc.,"Cupertino, CA",,0000320193,1976-04-01
ABNB,Airbnb,"San Francisco, CA",,0001559720,2008-08-01
ADBE,Adobe Inc.,"San Jose, CA",,0000796343,1982-12-01
...
```

## nasdaq100_price_change.csv

```py
symbol,1D,5D,1M,3M,6M,ytd,1Y,3Y,5Y,10Y,max
AAPL,-1.7254,-8.30086,-6.20411,3.042,15.64824,42.99992,8.47941,60.96299,245.42031,976.99441,139245.53954
ABNB,2.1617,-2.21919,9.88336,19.43286,19.64241,68.66902,23.64013,-1.04347,-1.04347,-1.04347,-1.04347
ADBE,0.5409,-1.77817,9.16191,52.0465,38.01522,57.22723,21.96206,17.83037,109.05718,1024.69214,251030.66399
ADI,0.9291,-4.03352,2.58486,3.65887,5.01602,17.02062,8.09735,63.42847,92.81874,286.77518,26012.63736
...
```

In [1]:
import os
import pandas as pd
from openai import OpenAI

ai_bridge = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

df_main = pd.read_csv("nasdaq100_CA.csv")
df_performance = pd.read_csv("nasdaq100_price_change.csv")

nasdaq_merged = pd.merge(df_main, df_performance[["symbol", "ytd"]], on="symbol")

print(nasdaq_merged.head())

for ticker in nasdaq_merged["symbol"]:
    classification_query = (
        f"Identify the sector for {ticker} from this list: Technology, Consumer Cyclical, "
        "Industrials, Utilities, Healthcare, Communication, Energy, Consumer Defensive, "
        "Real Estate, Financial. Return the name only."
    )
    
    api_out = ai_bridge.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": classification_query}],
        temperature=0
    )
    
    label = api_out.choices[0].message.content
    nasdaq_merged.loc[nasdaq_merged["symbol"] == ticker, "Industry_Group"] = label

print(nasdaq_merged["Industry_Group"].value_counts())

final_analysis_query = (
    f"Based on this data: {nasdaq_merged}, summarize YTD performance for Nasdaq-100 "
    "firms in CA. Suggest the top two sectors and list at least two stocks for each."
)

final_response = ai_bridge.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": final_analysis_query}],
    temperature=0
)

investment_summary = final_response.choices[0].message.content
print(investment_summary)

  symbol               name        headQuarter  ...      cik     founded       ytd
0   AAPL         Apple Inc.      Cupertino, CA  ...   320193  1976-04-01  42.99992
1   ABNB             Airbnb  San Francisco, CA  ...  1559720  2008-08-01  68.66902
2   ADBE         Adobe Inc.       San Jose, CA  ...   796343  1982-12-01  57.22723
3   ADSK           Autodesk     San Rafael, CA  ...   769397  1982-01-30  10.02701
4   AMAT  Applied Materials    Santa Clara, CA  ...     6951  1967-11-10  55.46366

[5 rows x 7 columns]
Technology            26
Healthcare             5
Communication          2
Consumer Cyclical      2
Real Estate            1
Consumer Defensive     1
Financial              1
Name: Industry_Group, dtype: int64
Based on the data provided, the top two sectors for Nasdaq-100 firms in California based on YTD performance are Technology and Healthcare.

Top two stocks in the Technology sector:
1. Nvidia (NVDA) - YTD performance: 217.27%
2. Meta Platforms (META) - YTD performance: 1